# Farsi floret vectors on Colab (Weasel pipeline)

Persian Wikipedia -> floret vectors on GPU -> an installable spaCy model you can `spacy.load("fa_floret")` later.

**Runtime -> Change runtime type -> T4 GPU** first.

Adapted from explosion/projects `floret_wiki_oscar_vectors`, restored to its XML-dump + wikiextractor(`--no-templates`) corpus route: the `wikimedia/wikipedia` HF text field still carries raw `[[wikilinks]]`, `{{templates}}` and HTML that floret would otherwise learn as vocabulary. OSCAR stays dropped (`unshuffled_deduplicated_fa` is gated and fails to load unauthenticated). Code ships as an uploaded zip, not a clone.

In [ ]:
# 1 -- GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch, os; print('cuda', torch.cuda.is_available(), '| vCPUs', os.cpu_count())

## Upload the code
Build the bundle locally first, once, from the repo root:
```
bash colab/make_bundle.sh
```
That zips the package, the `colab/farsi_weasel` weasel pipeline, and the tests into `floret_torch_bundle.zip`. Upload it below -- no GitHub PAT needed.

In [ ]:
# 2 -- upload the bundle and unpack it into /content
from google.colab import files
files.upload()
!unzip -o -q floret_torch_bundle.zip -d /content
%cd /content
!ls floret_torch colab/farsi_weasel

In [ ]:
# 3 -- dependencies. `-e .` puts floret_torch on the path so the weasel
#      commands can run from the project dir. torch is already present.
!pip -q install -e . 2>&1 | tail -2
!pip -q install spacy weasel datasets floret mmh3 wikiextractor 2>&1 | tail -2
import spacy, weasel, floret_torch; print('spacy', spacy.__version__, '| ok')

In [ ]:
# 4 -- parity gate: our hashing must match CPU floret byte for byte
!python -m pytest tests/test_parity.py -q

## Build the corpus
Three stages, matching explosion/projects: download the `fawiki` XML dump, run WikiExtractor with `--no-templates` (strips `[[wikilinks]]`, `{{templates}}`, HTML *before* tokenization), then tokenize the cleaned JSONL. `max_texts` caps how many articles `tokenize-wikipedia` reads from the extracted JSONL.

**Disk budget:** dump download ~1.3 GB, wikiextractor JSONL a few GB, tokenized corpus well under 1 GB at 100k-400k articles. Budget >=20 GB free in `/content` (a standard Colab runtime has far more). `cleanup-corpus` deletes the dump and extracted JSONL right after tokenizing, so only the small corpus file remains.

| max_texts | tokens | rough wall time |
|---|---|---|
| 50,000 | ~20M | few min |
| 100,000 | ~41M | ~10 min |
| 400,000 | ~163M | ~30 min |

100k is a good first run; 400k gives noticeably better vectors.

In [ ]:
# 5 -- download the Persian Wikipedia XML dump (~1.3 GB)
%cd /content/colab/farsi_weasel
!python -m weasel run download-wikipedia .
!du -h /content/downloaded/*

In [ ]:
# 6 -- extract with WikiExtractor. --no-templates strips [[wikilinks]],
#      {{templates}} and HTML before tokenization ever sees them.
!python -m weasel run extract-wikipedia . --vars.n_process 2
!du -h /content/extracted/AA/wiki_00

In [ ]:
# 7 -- tokenize -> one sentence per line, then reclaim the dump + extracted
#      JSONL (cleanup-corpus) so only the small corpus file stays on disk
!python -m weasel run tokenize-wikipedia . --vars.max_texts 100000
!python -m weasel run cleanup-corpus .
!df -h /content | tail -1
!wc -l /content/corpus/fa.txt && head -2 /content/corpus/fa.txt

In [ ]:
# 8 -- train on GPU. Keep --batch in 4096-16384: the input-row gradient is
#      un-normalized (as fastText does it), so an oversized batch accumulates
#      hundreds of full gradients onto one row and diverges.
!python -m weasel run train-vectors-gpu . --vars.vector_epoch 5 --vars.vector_batch 8192

In [ ]:
# 9 -- wrap into a spaCy pipeline and build an installable wheel
!python -m weasel run init-vectors .
!python -m weasel run package-vectors .
!find /content/packages -name '*.whl'

## Persist it — Colab wipes `/content` when the runtime recycles
This copies the `.floret` table, the `.vec` file, the zipped pipeline and the wheel to Drive. The `.floret` table is the portable artifact: it rebuilds the pipeline anywhere via `spacy init vectors --mode floret`.

In [ ]:
# 10 -- save to Drive (mounts it if needed)
!python -m weasel run save-to-drive .

In [ ]:
# 11 -- install and use, exactly as you would any spaCy model
import glob
whl = glob.glob('/content/packages/**/*.whl', recursive=True)[0]
!pip -q install --force-reinstall {whl}

import spacy
nlp = spacy.load('fa_floret')
print(nlp.vocab.vectors.shape, nlp.vocab.vectors.mode)

doc = nlp('کتاب‌های خوبی در دانشگاه تهران نوشتند')
for t in doc:
    print(f'{t.text:<14} norm={t.vector_norm:.3f} has_vector={t.has_vector}')

print('sim:', nlp('دانشگاه').similarity(nlp('دانشکده')))
# floret has no fixed vocab: a never-seen word still gets a real vector
print('unseen:', nlp('واژه‌ای‌که‌هرگزدیده‌نشده')[0].has_vector)

In [ ]:
# 12 -- neighbour sanity check
!python -m weasel run evaluate-vectors .

## Using it later, in a fresh session
Nothing above needs re-running. Either install the wheel from Drive, or rebuild from the `.floret` table.

In [ ]:
# 13 -- option A: install the wheel straight from Drive
from google.colab import drive; drive.mount('/content/drive')
!pip -q install /content/drive/MyDrive/floret_fa/fa_floret-0.1.0-py3-none-any.whl
import spacy; nlp = spacy.load('fa_floret'); print(nlp.vocab.vectors.shape)

In [ ]:
# 14 -- option B: rebuild the pipeline from the portable .floret table
!python -m spacy init vectors fa \
  /content/drive/MyDrive/floret_fa/fa.floret /content/fa_pipeline \
  --mode floret --name floret
import spacy; nlp = spacy.load('/content/fa_pipeline'); print(nlp.vocab.vectors.shape)

## Using the vectors to train a real component
Vectors are an input to training, not a model. To train a Persian tagger/parser on top of them, point the config's `[initialize] vectors` at the pipeline directory and set `[components.tok2vec] include_static_vectors = true`:

```bash
python -m spacy init config config.cfg --lang fa --pipeline tagger,parser --optimize accuracy
python -m spacy train config.cfg \
  --paths.train train.spacy --paths.dev dev.spacy \
  --initialize.vectors /content/vectors/fa_floret \
  --components.tok2vec.model.embed.include_static_vectors true \
  --gpu-id 0
```

Persian treebanks for `train.spacy`: `UD_Persian-Seraji` or `UD_Persian-PerDT`, converted with `spacy convert file.conllu ./ --converter conllu`.